In [185]:
import numpy as np
import pandas as pd
import datetime
import yfinance as yf
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

# For importing universal scripts
import sys
import os
# Go up two levels from the subfolder
sys.path.append(os.path.abspath(".."))
from indicators_returns import final_df #Universal script for indicator set and actuals
import importlib
import indicators_returns
importlib.reload(indicators_returns)
from indicators_returns import final_df
import gc
from sklearn.metrics import (fbeta_score, accuracy_score, f1_score, 
                             confusion_matrix, balanced_accuracy_score, recall_score, matthews_corrcoef, precision_score)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split, StratifiedKFold
from xgboost import XGBClassifier
import math
import pickle
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit

tags = pd.read_csv('../Indicator_Selection_Pipeline/Finalization/tags_cons.csv') 

def extract(ticker, returns, lb, cat_cols_all, windows=[10, 25]):

    import os
    # point this at an existing directory in your project
    os.chdir('/Users/brettchase/Documents/Fracturion/Feature_Set_Construction_Testing')
    
    df = final_df(ticker, returns, lb)
    df = df.iloc[:-101].replace([np.inf, -np.inf], 0)#

    df = df.sort_index(ascending=True)
    # Exponential Moving Average
    ema_cols = {
        f"{col}_EMA{w}": df[col].ewm(span=w, adjust=False).mean()
        for w in windows
        for col in cat_cols_all
    }
    # 3) merge them back into one dict
    new_cols = {**ema_cols}

    # 4) concatenate onto your original df
    df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)    
    df = df.sort_index(ascending=False)
    
    return df

lb = 8
returns = [5, 10, 15, 20, 25, 30]
raw_all = tags['Indicator'][tags['Type'] == 'Raw']
df = extract('QQQ', returns, lb, raw_all, windows=[10,25])

# By category × velocity
duration_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
duration_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
duration_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
trend_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
trend_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
trend_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
trend_ratio_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
trend_ratio_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
trend_ratio_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
volatility_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
volatility_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
volatility_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
momentum_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
momentum_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
momentum_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
lag_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()

,Date,Close,Past_Return_10,Past_Return_25,wr_100_10,wr_100_25,wr_50_10
2011,2025-10-02,605.729980,1,1,0.87,0.99,0.76
2010,2025-10-01,603.250000,1,1,0.87,0.99,0.76
2009,2025-09-30,600.369995,1,1,0.87,0.99,0.76
2008,2025-09-29,598.729980,1,1,0.87,0.99,0.76
2007,2025-09-26,595.969971,1,1,0.87,0.99,0.76
2006,2025-09-25,593.530029,1,1,0.87,0.99,0.76
2005,2025-09-24,596.099976,1,1,0.87,0.99,0.76
2004,2025-09-23,598.200012,1,1,0.87,0.98,0.76
2003,2025-09-22,602.200012,1,1,0.87,0.97,0.76
2002,2025-09-19,598.655945,1,1,0.87,0.96,0.76


In [167]:
def past_return(df, days):

    future_return = (df['Close'] - df['Close'].shift(-days)) / df['Close']

    if days >= 100:
        df[f'Return_{days}'] = np.where(
            future_return > 0.001, 1,
            np.where(future_return < -0.001, 0, np.nan)
        )
    else:
        df[f'Past_Return_{days}'] = (future_return > 0).astype(int)

    return df

dft = df.copy()
for r in returns: 
    new_df = past_return(dft, r)

In [171]:
new_df[['Date', 'Close', 'Past_Return_25']].head(50)

,Date,Close,Past_Return_25
2011,2025-10-02,605.729980,1
2010,2025-10-01,603.250000,1
2009,2025-09-30,600.369995,1
2008,2025-09-29,598.729980,1
2007,2025-09-26,595.969971,1
2006,2025-09-25,593.530029,1
2005,2025-09-24,596.099976,1
2004,2025-09-23,598.200012,1
2003,2025-09-22,602.200012,1
2002,2025-09-19,598.655945,1


In [98]:
windows = [50, 100, 200]
returns = [5, 10, 15, 20, 25, 30]

for r in returns:

    for window in windows:

        # use standard rolling for last 50 values
        df[f'wr_{window}_{r}'] = df[f"Return_{r}"].iloc[r:].sort_index(ascending=True).rolling(window).sum() / window
        df[f'wr_{window}_{r}'] = df[f'wr_{window}_{r}'].bfill()

In [62]:
tickerSymbol = 'QQQ'
lb = 8

# Get data on this ticker
tickerData = yf.Ticker(tickerSymbol)
start_date = (datetime.today() - relativedelta(years=lb)).strftime('%Y-%m-%d')

tickerDf = tickerData.history(period='1d', start=start_date)

# Resetting the index will turn the Date index into a column
df_sma = tickerDf.reset_index()[['Date', 'Close', 'High', 'Low', 'Volume']]

# Convert the Date column to 'YYYY-MM-DD' format (if not already)
df_sma['Date'] = pd.to_datetime(df_sma['Date']).dt.strftime('%Y-%m-%d')

df = df_sma.sort_index(ascending=True)

In [63]:
df

,Date,Close,High,Low,Volume
0,2017-10-02,138.048904,138.608379,137.432521,27351300
1,2017-10-03,138.342834,138.504038,137.925594,19715200
2,2017-10-04,138.475601,138.769562,137.944573,26381100
3,2017-10-05,139.831665,139.907514,138.835980,31562400
4,2017-10-06,140.021332,140.021332,139.357547,23561100
...,...,...,...,...,...
2007,2025-09-26,595.969971,596.299988,591.059998,54337400
2008,2025-09-29,598.729980,602.049988,597.409973,48332900
2009,2025-09-30,600.369995,600.710022,596.099976,46533800
2010,2025-10-01,603.250000,603.789978,596.340027,46899600
